In [77]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [78]:
#DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"
DATA_PATH = "D:/Google Drive/Work/ITMO/data/schreiber_full/"

In [79]:
from mhcgnomes import parse

# read total data

source_total_df =  pd.read_csv("{}schreiber_total_data_full.csv".format(DATA_PATH),sep=";")
ALLELES = list(set(source_total_df['allele']))
per_allele_mt_df =dict()
per_allele_wt_df = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    df_mt = pd.read_csv("{}schreiber_data_full_{}_mt.csv".format(DATA_PATH,target_allele_name),sep=";")
    df_wt = pd.read_csv("{}schreiber_data_full_{}_wt.csv".format(DATA_PATH,target_allele_name),sep=";") 
    
    per_allele_mt_df[allele_name] = df_mt
    per_allele_wt_df[allele_name] = df_wt
    
    assert 'peptide' in df_mt.columns
    assert 'length' in df_mt.columns
    assert RESULT_COLUMNS['id'] in df_mt.columns
    assert 'peptide' in df_wt.columns
    assert 'length' in df_wt.columns
    assert RESULT_COLUMNS['id'] in df_wt.columns
    
    
def defineOrganism(allele_name):
    parse_result = parse(allele_name)
    if parse_result.is_mouse:
        return "Mouse"
    elif parse_result.is_human:
        return "Human"
    else:
        print("organism not found")
        return None


def defineClass(allele_name):
    parse_result = parse(allele_name)
    if parse_result.has_mhc_class:
        if parse_result.is_class1:
            return 'I'
        elif parse_result.is_class2:
            return 'II'
        else:
            print("Not class I or class II???!")
    else:
        print("Result without class")
    return None
ALLELES

['H-2-IEd', 'H-2-Ld', 'H-2-IAd', 'H-2-Dd', 'H-2-Kd']

# 1. Read all the predictions

In [60]:
import os


In [81]:
per_method_columns = {
    'NetMHC': ['netMHC-3.4_score', 'netMHC-3.4_affinity', 'netMHC-4.0_score',  'netMHC-4.0_affinity','netMHC-4.0_percentile_rank', \
               'netMHCpan-4.1_score_ba', 'netMHCpan-4.1_score_el', 'netMHCpan-4.1_affinity', 'netMHCpan-4.1_percentile_rank_ba', 'netMHCpan-4.1_percentile_rank_el'  ],
    'netMHC-3.4': [ 'score', 'affinity'],
    'netMHC-4.0':[ 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':[ 'score_ba', 'score_el', 'affinity', 'percentile_rank_ba', 'percentile_rank_el'],
    'mhcflurry':['affinity', 'percentile_rank_ba', 'processing_score', 'score_el','percentile_rank_el' ],
    'mhcnuggets':[ 'score', 'affinity', 'percentile_rank'],
    'acme':['score'],
    'puffin':['score', 'epistemic_var', 'aleatoric_var', 'binding_likelihood', 'percentile_rank'],
    'mhcseqnet':['score'],
    'mhcattnnet':[ 'score'],
    'ghmm': ['HMMB', 'percentile_rank']
}



In [82]:
PREDICTION_PATH = "D:/Google Drive/Work/ITMO/MHC-Epitop-Prediction/prediction_results/schreiber_full/"
methods =  next(os.walk(PREDICTION_PATH))[1] # read all the folders in a prediction path
methods.remove('NetMHC')
methods.insert(0, 'NetMHC')

#['NetMHC', 'puffin', 'mhcnuggets', 'acme', 'mhcseqnet', 'mhcattnnet']

per_allele_per_method_predictions = dict()

less_is_good_columns = {'affinity','percentile_rank', 'netMHC-3.4_affinity', 'netMHC-4.0_affinity', 'percentile_rank_el', 'percentile_rank_ba',
                                    'netMHCpan-4.1_affinity', 'netMHCpan-4.1_percentile_rank_ba', 
                                    'netMHCpan-4.1_percentile_rank_el', 'netMHC-4.0_percentile_rank', 'epistemic_var', 'aleatoric_var' }


for allele_name in ALLELES:
    print(f"{allele_name}:", end=" ")
    per_allele_per_method_predictions[allele_name] = dict()
    for method in methods:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
        prediction_df = pd.read_csv(f"{PREDICTION_PATH}{method}/{method}_predictions_allele_{target_allele_name}.csv", sep=';')

        #add diff 

        print(f"{method}", end=" ")
        for score_column in per_method_columns[method]:
            result_score_column = f"{method}_{score_column}" if not (method == 'NetMHC') else score_column
            wt_scores = prediction_df[result_score_column + '_wt'].values
            mt_scores = prediction_df[result_score_column + '_mt'].values
            wt_peptide =  prediction_df['peptide_wt'].values
            # for these columns mt shoud be less than wt. In this case wt/mt will be > 1 and this is good. 
            if score_column in less_is_good_columns:
                
                advantage = np.array([wt_sc/(mt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            # for these columns mt shoud be greater than wt. In this case mt/wt will be > 1 and this is good. 
            else:
                advantage = np.array([mt_sc/(wt_sc + 0.0000000001) if not peptide == np.nan else 2  for wt_sc,mt_sc, peptide in zip(wt_scores,mt_scores,wt_peptide)])
            prediction_df[f"{result_score_column}_advantage"] = advantage



        per_allele_per_method_predictions[allele_name][method] = prediction_df
    print()
            

H-2-IEd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-Ld: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-IAd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-Dd: NetMHC ghmm mhcflurry mhcnuggets puffin 
H-2-Kd: NetMHC ghmm mhcflurry mhcnuggets puffin 


In [84]:
per_allele_per_method_predictions[ALLELES[1]]['NetMHC']

,source_sequence_name,allele,peptide_mt,length,netMHC-3.4_score_mt,netMHC-3.4_affinity_mt,netMHC-3.4_percentile_rank_mt,netMHC-4.0_score_mt,netMHC-4.0_affinity_mt,netMHC-4.0_percentile_rank_mt,...,netMHC-3.4_score_advantage,netMHC-3.4_affinity_advantage,netMHC-4.0_score_advantage,netMHC-4.0_affinity_advantage,netMHC-4.0_percentile_rank_advantage,netMHCpan-4.1_score_ba_advantage,netMHCpan-4.1_score_el_advantage,netMHCpan-4.1_affinity_advantage,netMHCpan-4.1_percentile_rank_ba_advantage,netMHCpan-4.1_percentile_rank_el_advantage
0,seq1,H-2-Ld,ALSGVSVYF,9,0.037,33498.0,NaN,0.041,32036.15,12.0,...,0.435294,0.592991,0.611940,0.759600,0.541667,0.926477,0.553739,0.900962,0.843995,0.615784
1,seq2,H-2-Ld,QPAAAPAAAL,10,0.338,1289.0,NaN,0.149,9991.85,2.0,...,0.882507,0.613654,1.056738,1.089636,1.250000,1.100174,0.958146,1.208604,1.226221,0.956021
2,seq3,H-2-Ld,LSGVSVYF,8,0.070,23375.0,NaN,0.053,28269.03,9.0,...,0.933333,0.948449,0.688312,0.768628,0.611111,0.975785,0.473203,0.982838,0.957528,0.617716
3,seq4,H-2-Ld,QVLGGVNL,8,0.047,30194.0,NaN,0.084,20206.55,5.0,...,0.701493,0.802279,1.909091,1.538372,2.200000,0.921650,0.843375,0.956670,0.890329,0.893870
4,seq5,H-2-Ld,AAAPAAAL,8,0.038,33038.0,NaN,0.014,42995.14,37.0,...,1.310345,1.102064,0.202899,0.551502,0.175676,1.004657,1.990724,1.003599,1.008941,1.740253
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8733,seq17637,H-2-Ld,REPAPEWPSIN,11,0.006,46727.0,NaN,0.110,15260.97,3.5,...,1.000000,1.002675,0.723684,0.633190,0.571429,0.952766,1.305873,0.964163,0.915400,1.180705
8734,seq17638,H-2-Ld,DKGKIEDTGGS,11,0.005,47178.0,NaN,0.004,47994.24,95.0,...,1.000000,0.998961,1.000000,0.999286,1.000000,0.987747,0.999900,0.999556,0.999651,1.000000
8735,seq17639,H-2-Ld,NDFSVLPLPN,10,0.006,46677.0,NaN,0.004,47756.99,95.0,...,0.857143,0.998050,1.000000,0.999146,0.947368,1.123433,1.999988,1.031495,1.121004,1.162494
8736,seq17640,H-2-Ld,EPAPEWPSINL,11,0.368,928.0,NaN,0.291,2139.38,0.6,...,0.997290,0.992457,0.803867,0.467645,0.500000,1.007832,1.707622,1.011054,1.017058,1.588421


# 2. Merge predictions

In [85]:
per_allele_predictions = dict()
for allele_name in ALLELES:
    list_for_allele = list(per_allele_per_method_predictions[allele_name].values())

    result_df = list_for_allele[0].copy()
    for df_to_merge in list_for_allele[1:]:
        result_df = result_df.merge(df_to_merge, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'], how='left')
    result_df = result_df[['source_sequence_name','allele', 'length'] + [column for column in result_df.columns if column.endswith('mt')] 
                          + [column for column in result_df.columns if column.endswith('_advantage')] 
                          + [column for column in result_df.columns if column.endswith('wt')]] 
    per_allele_predictions[allele_name] = result_df


In [86]:
all_methods = list(set([item.split('_')[0] for item in result_df.columns if  item.split('_')[0] not in ['allele', 'peptide', 'length', 'source']]))
all_methods

['mhcnuggets',
 'ghmm',
 'mhcflurry',
 'netMHC-3.4',
 'puffin',
 'netMHCpan-4.1',
 'netMHC-4.0',
 'netMHCIIpan-4.0']

In [87]:
from config import PER_ORGANISM_PER_CLASS_ALLELES

unsupported_alleles = {method : set() for method in all_methods}

def add_unsupported_allele(method, alleles):
    methods_set = set(all_methods)
    if (method in methods_set):
        unsupported_alleles[method].update(alleles)

for allele_name in ALLELES:
    if defineClass(allele_name) == 'II':
        add_unsupported_allele('netMHCpan-4.1', [allele_name])
        add_unsupported_allele('netMHC-3.4', [allele_name])
        add_unsupported_allele('netMHC-4.0', [allele_name])
        add_unsupported_allele('acme', [allele_name])
        add_unsupported_allele('mhcflurry', [allele_name])
    else:
        add_unsupported_allele('netMHCIIpan-4.0', [allele_name])
        
    if not defineOrganism(allele_name) == 'Human':
        add_unsupported_allele('mhcseqnet', [allele_name])
        add_unsupported_allele('mhcattnnet', [allele_name])
        add_unsupported_allele('acme', [allele_name])

#special cases
add_unsupported_allele('acme',["HLA-C*04:01", "HLA-C*03:04"])
add_unsupported_allele('puffin',["HLA-C*04:01"])



def drop_unsupported_columns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        for method in all_methods:
            if allele_name in unsupported_alleles[method]:
                columns_to_drop = [col.strip() for col in df.columns if col.startswith(method)] + ['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt']
                df = df.drop(columns=columns_to_drop, errors='ignore')    
            per_allele_df[allele_name] = df
        
            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return per_allele_df


def reorder_columns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        general_info = [RESULT_COLUMNS['id'], 'allele', 'length']
        
        
        percentile_columns_mt = [column for column in df.columns if column.find('percentile')>=0 and column.endswith('mt')]
        percentile_columns_wt = [column for column in df.columns if column.find('percentile')>=0 and column.endswith('wt')]
        percentile_columns = percentile_columns_mt + percentile_columns_wt
        not_percentile_columns = [column for column in df.columns if column not in percentile_columns and  column not in general_info and column not in ['peptide_wt', 'peptide_mt']]
        
        columns_ordered = list()
        columns_ordered.extend(general_info)
        columns_ordered.append('peptide_mt')
        columns_ordered.extend(percentile_columns_mt)
        columns_ordered.append('peptide_wt')
        columns_ordered.extend(percentile_columns_wt)
        columns_ordered.extend(not_percentile_columns)

        
        per_allele_df[allele_name] = df[columns_ordered]
            #df = df.drop(columns=['netMHC-3.4_percentile_rank_mt', 'netMHC-3.4_percentile_rank_wt'])
    return per_allele_df
per_allele_predictions = reorder_columns(drop_unsupported_columns(per_allele_predictions))

# Add source info

In [88]:
source_total_columns = ['VAR_ID', 'mutant_id', 'transcript_id', 'peptide_offset',
                        'A20_RGT', 'A20_RDP', 'A20_RAD_REF',
                        'A20_RAD_ALT', 'A20_RAF_ALT', 'A20_RPKM_mut',  'source_sequence_name', 
                        'peptide_wt', 'peptide_mt' , 'allele',  'length'
                       ]
source_total_df = source_total_df[source_total_columns]
source_total_df

,VAR_ID,mutant_id,transcript_id,peptide_offset,A20_RGT,A20_RDP,A20_RAD_REF,A20_RAD_ALT,A20_RAF_ALT,A20_RPKM_mut,source_sequence_name,peptide_wt,peptide_mt,allele,length
0,4_147755805_T/A,Zfp984-K196N,ENSMUST00000105734,188,T/A,249,95,154,0.618474,5.234282,seq0,KYLTQREKL,KYLTQRENL,H-2-Kd,9
1,18_53706875_A/C,Cep120-L771R,ENSMUST00000049811,765,A/C,259,110,149,0.575290,3.492483,seq6,VYQVELERL,VYQVERERL,H-2-Kd,9
2,11_69804901_G/C,Tmem102-P82A,ENSMUST00000051025,79,G/C,90,54,36,0.400000,1.645296,seq11,RFPPQAELL,RFAPQAELL,H-2-Kd,9
3,3_104800372_T/C,Mov10-D635G,ENSMUST00000166979,627,T/C,181,82,99,0.546961,1.347878,seq14,APSNSGADL,APSNSGAGL,H-2-Kd,9
4,11_60031753_G/A,Pemt-A55V,ENSMUST00000102693,50,G/A,21,15,6,0.285714,0.061122,seq15,SFVAAVITI,SFVAVVITI,H-2-Kd,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130391,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,233,A/G,939,0,939,1.000000,10.112060,seq92523,MCNSSCMGGMNR,MCNGSCMGGMNR,H-2-IAd,12
130392,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,232,A/G,939,0,939,1.000000,10.112060,seq92527,YMCNSSCMGGMNR,YMCNGSCMGGMNR,H-2-IAd,13
130393,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,231,A/G,939,0,939,1.000000,10.112060,seq92528,KYMCNSSCMGGMN,KYMCNGSCMGGMN,H-2-IAd,13
130394,11_69589186_A/G,Trp53-S237G,ENSMUST00000108658,231,A/G,939,0,939,1.000000,10.112060,seq92534,KYMCNSSCMGGM,KYMCNGSCMGGM,H-2-IAd,12


In [89]:
class_I_dfs = list()
class_II_dfs = list()
for allele_name in ALLELES:
    df_allele_source = source_total_df.loc[source_total_df['allele'] == allele_name].copy()
    df_predictions = per_allele_predictions[allele_name]
    df_result = df_predictions.merge(df_allele_source, on=['source_sequence_name', 'peptide_mt', 'length', 'peptide_wt', 'allele'])
    if defineClass(allele_name) == 'I':
        class_I_dfs.append(df_result)
    elif defineClass(allele_name) == 'II':
        class_II_dfs.append(df_result)
        
class_I_total_df = pd.concat(class_I_dfs)
class_II_total_df = pd.concat(class_II_dfs)
    
    
    

In [90]:
class_I_total_df

,source_sequence_name,allele,length,peptide_mt,netMHC-4.0_percentile_rank_mt,netMHCpan-4.1_percentile_rank_ba_mt,netMHCpan-4.1_percentile_rank_el_mt,ghmm_percentile_rank_mt,mhcflurry_percentile_rank_ba_mt,mhcflurry_percentile_rank_el_mt,...,VAR_ID,mutant_id,transcript_id,peptide_offset,A20_RGT,A20_RDP,A20_RAD_REF,A20_RAD_ALT,A20_RAF_ALT,A20_RPKM_mut
0,seq1,H-2-Ld,9,ALSGVSVYF,12.0,3.314,2.889,0.00275,6.68775,5.149212,...,8_106195970_C/T,Slc7a6-P463S,ENSMUST00000034378,457,C/T,432,242,190,0.439815,3.979889
1,seq2,H-2-Ld,10,QPAAAPAAAL,2.0,1.167,0.955,0.00375,0.20225,1.703804,...,4_119294333_G/C,Ybx1-P11A,ENSMUST00000079644,8,G/C,7165,3612,3553,0.495883,91.274678
2,seq3,H-2-Ld,8,LSGVSVYF,9.0,12.314,12.012,0.00475,45.40725,27.617391,...,8_106195970_C/T,Slc7a6-P463S,ENSMUST00000034378,458,C/T,432,242,190,0.439815,3.979889
3,seq4,H-2-Ld,8,QVLGGVNL,5.0,19.212,9.658,0.00925,28.15025,46.224674,...,11_106048205_C/G,Dcaf7-R158G,ENSMUST00000058438,153,C/G,911,0,911,1.000000,44.449302
4,seq5,H-2-Ld,8,AAAPAAAL,37.0,10.178,3.873,0.01125,5.79625,9.849457,...,4_119294333_G/C,Ybx1-P11A,ENSMUST00000079644,10,G/C,7165,3612,3553,0.495883,91.274678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8733,seq14195,H-2-Kd,8,PTTVCQPN,37.0,92.530,99.167,99.94325,93.94425,99.286603,...,2_4893305_G/C,Sephs1-A176P,ENSMUST00000027973,175,G/C,429,221,208,0.484848,4.831458
8734,seq14197,H-2-Kd,8,PRLHHSSR,65.0,90.631,78.500,99.94925,77.64425,99.286603,...,X_71939448_T/G,Gm1141-L342R,ENSMUST00000101495,340,T/G,4,0,4,1.000000,0.039319
8735,seq14200,H-2-Kd,11,PPRLHHSSRSC,60.0,85.753,70.833,99.90475,70.53175,99.286603,...,X_71939448_T/G,Gm1141-L342R,ENSMUST00000101495,339,T/G,4,0,4,1.000000,0.039319
8736,seq14201,H-2-Kd,8,PNNISAIS,75.0,92.988,88.333,99.96775,93.25975,99.286603,...,4_107047386_G/C,Ssbp3-G353A,ENSMUST00000030367,347,G/C,424,224,200,0.471698,1.247553


In [91]:
sum([len(df) for df in per_allele_mt_df.values()])

130396

In [92]:
len(class_I_total_df) + len(class_II_total_df)

130396

In [93]:
len(source_total_df)

130396

## Save total pred to file

In [94]:
# for decoy_key in DECOY_PREFIXES:
for allele_name in ALLELES:
    total_df = per_allele_predictions[allele_name]
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    total_df.to_csv("{}total_predictions_allele_{}.csv".format(PREDICTION_PATH,target_allele_name),sep=";", index=False)

In [95]:
class_I_total_df.to_csv("{}total_predictions_class_I.csv".format(PREDICTION_PATH),sep=";", index=False)
class_II_total_df.to_csv("{}total_predictions_class_II.csv".format(PREDICTION_PATH),sep=";", index=False)